# Ejercicios ensembling
En este ejercicio vas a realizar prediciones sobre un dataset de ciudadanos indios diabéticos. Se trata de un problema de clasificación en el que intentaremos predecir 1 (diabético) 0 (no diabético).

### 1. Carga las librerias que consideres comunes al notebook

In [1]:
import pandas as pd
import numpy as np
import matplotlib as mpl
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import GradientBoostingClassifier

### 2. Lee los datos de [esta direccion](https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv)
Los nombres de columnas son:
```Python
names = ['preg', 'plas', 'pres', 'skin', 'test', 'mass', 'pedi', 'age', 'class']
```

In [2]:
names = ['preg', 'plas', 'pres', 'skin', 'test', 'mass', 'pedi', 'age', 'class']
df = pd.read_csv('https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv', names=names)  
df.head()

,preg,plas,pres,skin,test,mass,pedi,age,class
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [3]:
df.describe()

,preg,plas,pres,skin,test,mass,pedi,age,class
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,120.894531,69.105469,20.536458,79.799479,31.992578,0.471876,33.240885,0.348958
std,3.369578,31.972618,19.355807,15.952218,115.244002,7.884160,0.331329,11.760232,0.476951
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,62.000000,0.000000,0.000000,27.300000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,30.500000,32.000000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


### 3. Bagging
Para este apartado tendrás que crear un ensemble utilizando la técnica de bagging ([BaggingClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.BaggingClassifier.html)), mediante la cual combinarás 100 [DecisionTreeClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html). Recuerda utilizar también [cross validation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.KFold.html) con 10 kfolds.

**Para este apartado y siguientes, no hace falta que dividas en train/test**, por hacerlo más sencillo. Simplemente divide tus datos en features y target.

Establece una semilla

In [4]:
X = df.drop('class', axis=1).values  # Todas las features (8 columnas)
y = df['class'].values

In [5]:
from sklearn.model_selection import cross_val_score

estimator = DecisionTreeClassifier(max_depth=3,random_state=42)

bag_clf = BaggingClassifier(
    estimator = estimator,
    n_estimators=100, # Cantidad de árboles
    max_samples=100, # Muestras utilizadas en bootstrapping
    bootstrap=True, # Usamos bootstrapping: muestreo con reemplazo.
    max_features = 3, # Features que utiliza en el bootstrapping. Cuanto más bajo, mejor generalizará y menos overfitting
    random_state=42)

# Cross validation con 10 kfolds
cv_scores = cross_val_score(bag_clf, X, y, cv=10, scoring='accuracy')
print(f"Scores CV: {cv_scores}")
print(f"Media: {cv_scores.mean():.4f}")

Scores CV: [0.76623377 0.75324675 0.77922078 0.7012987  0.74025974 0.72727273
 0.77922078 0.76623377 0.71052632 0.80263158]
Media: 0.7526


### 4. Random Forest
En este caso entrena un [RandomForestClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html) con 100 árboles y un `max_features` de 3. También con validación cruzada

In [6]:
rf_clf = RandomForestClassifier(
    n_estimators=100,
    max_features=3,
    random_state=42
)

# Cross validation con 10 kfolds
rf_cv_scores = cross_val_score(rf_clf, X, y, cv=10, scoring='accuracy')
print(f"Random Forest - Scores CV: {rf_cv_scores}")
print(f"Random Forest - Media: {rf_cv_scores.mean():.4f}")

Random Forest - Scores CV: [0.72727273 0.79220779 0.77922078 0.66233766 0.72727273 0.77922078
 0.83116883 0.87012987 0.71052632 0.80263158]
Random Forest - Media: 0.7682


### 5. AdaBoost
Implementa un [AdaBoostClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.AdaBoostClassifier.html) con 30 árboles.

In [7]:
ada_clf = AdaBoostClassifier(
    n_estimators=30,
    random_state=42
)

In [8]:
ada_cv_scores = cross_val_score(ada_clf, X, y, cv=10, scoring='accuracy')
print(f"AdaBoost - Scores CV: {ada_cv_scores}")
print(f"AdaBoost - Media: {ada_cv_scores.mean():.4f}")

AdaBoost - Scores CV: [0.71428571 0.76623377 0.77922078 0.63636364 0.7012987  0.77922078
 0.79220779 0.79220779 0.72368421 0.82894737]
AdaBoost - Media: 0.7514


### 6. GradientBoosting
Implementa un [GradientBoostingClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingClassifier.html) con 100 estimadores

In [9]:
# GradientBoosting con 100 estimadores
gb_clf = GradientBoostingClassifier(
    n_estimators=100,
    random_state=42
)

In [10]:
# Cross validation con 10 kfolds
gb_cv_scores = cross_val_score(gb_clf, X, y, cv=10, scoring='accuracy')
print(f"GradientBoosting - Scores CV: {gb_cv_scores}")
print(f"GradientBoosting - Media: {gb_cv_scores.mean():.4f}")

GradientBoosting - Scores CV: [0.72727273 0.80519481 0.79220779 0.63636364 0.74025974 0.77922078
 0.79220779 0.80519481 0.71052632 0.81578947]
GradientBoosting - Media: 0.7604


### 7. XGBoost
Para este apartado utiliza un [XGBoostClassifier](https://docs.getml.com/latest/api/getml.predictors.XGBoostClassifier.html) con 100 estimadores. XGBoost no forma parte de la suite de modelos de sklearn, por lo que tendrás que instalarlo con pip install

In [14]:
from xgboost import XGBClassifier
from xgboost import XGBClassifier

# XGBoost con 100 estimadores
xgb_clf = XGBClassifier(
    n_estimators=100,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)

# Cross validation con 10 kfolds
xgb_cv_scores = cross_val_score(xgb_clf, X, y, cv=10, scoring='accuracy')
print(f"XGBoost - Scores CV: {xgb_cv_scores}")
print(f"XGBoost - Media: {xgb_cv_scores.mean():.4f}")

c:\Users\rns_2\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\training.py:200: UserWarning: [21:49:15] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\rns_2\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\training.py:200: UserWarning: [21:49:15] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\rns_2\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\training.py:200: UserWarning: [21:49:15] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\rns_2\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\training.py:200: UserWarning: [21:49:15] WARNING: C:\actio

XGBoost - Scores CV: [0.68831169 0.74025974 0.75324675 0.67532468 0.74025974 0.75324675
 0.75324675 0.77922078 0.67105263 0.76315789]
XGBoost - Media: 0.7317


c:\Users\rns_2\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\training.py:200: UserWarning: [21:49:16] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


### 8. Primeros resultados
Crea un dataframe con los resultados y sus algoritmos, ordenándolos de mayor a menor

In [12]:
resultados = pd.DataFrame({
    'Algoritmo': ['Bagging', 'Random Forest', 'AdaBoost', 'GradientBoosting', 'XGBoost'],
    'Accuracy_Medio': [cv_scores.mean(), rf_cv_scores.mean(), ada_cv_scores.mean(), gb_cv_scores.mean(), xgb_cv_scores.mean()]
})

In [13]:
resultados = resultados.sort_values('Accuracy_Medio', ascending=False).reset_index(drop=True)
print("=" * 50)
print("RESULTADOS - Modelos sin hiperparámetros")
print("=" * 50)
print(resultados)

RESULTADOS - Modelos sin hiperparámetros
          Algoritmo  Accuracy_Medio
0     Random Forest        0.768199
1  GradientBoosting        0.760424
2           Bagging        0.752614
3          AdaBoost        0.751367
4           XGBoost        0.731733


### 9. Hiperparametrización
Vuelve a entrenar los modelos de nuevo, pero esta vez dividiendo el conjunto de datos en train/test y utilizando un gridsearch para encontrar los mejores hiperparámetros.

In [16]:
# Dividir datos en train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Diccionario para almacenar resultados
resultados_grid = {}

In [ ]:
from sklearn.model_selection import GridSearchCV
# 1. Bagging con GridSearch

bag_params = {
    'n_estimators': [50, 100, 150],
    'max_samples': [50, 100, 0.5],
    'max_features': [2, 3, 4]
}
bag_grid = GridSearchCV(BaggingClassifier(estimator=DecisionTreeClassifier(max_depth=3, random_state=42), 
                                           bootstrap=True, random_state=42),
                        bag_params, cv=5, scoring='accuracy')
bag_grid.fit(X_train, y_train)
resultados_grid['Bagging'] = {
    'best_params': bag_grid.best_params_,
    'best_score': bag_grid.best_score_,
    'test_score': bag_grid.score(X_test, y_test)
}

Buscando mejores hiperparámetros para Bagging...


In [22]:
# 2. Random Forest con GridSearch

rf_params = {
    'n_estimators': [50, 100, 150],
    'max_depth': [3, 5, 7, None],
    'max_features': [2, 3, 4]
}
rf_grid = GridSearchCV(RandomForestClassifier(random_state=42), rf_params, cv=5, scoring='accuracy')
rf_grid.fit(X_train, y_train)
resultados_grid['Random Forest'] = {
    'best_params': rf_grid.best_params_,
    'best_score': rf_grid.best_score_,
    'test_score': rf_grid.score(X_test, y_test)
}

In [23]:
# 3. GradientBoosting con GridSearch

gb_params = {
    'n_estimators': [50, 100],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 5, 7]
}
gb_grid = GridSearchCV(GradientBoostingClassifier(random_state=42), gb_params, cv=5, scoring='accuracy')
gb_grid.fit(X_train, y_train)
resultados_grid['GradientBoosting'] = {
    'best_params': gb_grid.best_params_,
    'best_score': gb_grid.best_score_,
    'test_score': gb_grid.score(X_test, y_test)
}

In [24]:
# 4. XGBoost con GridSearch

xgb_params = {
    'n_estimators': [50, 100],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 5, 7]
}
xgb_grid = GridSearchCV(XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss'), 
                        xgb_params, cv=5, scoring='accuracy')
xgb_grid.fit(X_train, y_train)
resultados_grid['XGBoost'] = {
    'best_params': xgb_grid.best_params_,
    'best_score': xgb_grid.best_score_,
    'test_score': xgb_grid.score(X_test, y_test)
}

c:\Users\rns_2\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\training.py:200: UserWarning: [21:57:55] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\rns_2\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\training.py:200: UserWarning: [21:57:55] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\rns_2\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\training.py:200: UserWarning: [21:57:55] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\rns_2\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\training.py:200: UserWarning: [21:57:55] WARNING: C:\actio

In [25]:
# Mostrar resultados
print("\n" + "=" * 60)
print("RESULTADOS - Modelos con GridSearchCV")
print("=" * 60)
for modelo, resultados in resultados_grid.items():
    print(f"\n{modelo}:")
    print(f"  Mejores parámetros: {resultados['best_params']}")
    print(f"  Best CV Score: {resultados['best_score']:.4f}")
    print(f"  Test Score: {resultados['test_score']:.4f}")


RESULTADOS - Modelos con GridSearchCV

Bagging:
  Mejores parámetros: {'max_features': 4, 'max_samples': 100, 'n_estimators': 150}
  Best CV Score: 0.7720
  Test Score: 0.8052

Random Forest:
  Mejores parámetros: {'max_depth': 7, 'max_features': 2, 'n_estimators': 50}
  Best CV Score: 0.7818
  Test Score: 0.7597

GradientBoosting:
  Mejores parámetros: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 50}
  Best CV Score: 0.7769
  Test Score: 0.7662

XGBoost:
  Mejores parámetros: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 50}
  Best CV Score: 0.7818
  Test Score: 0.7532


### 10. Conclusiones finales

In [26]:
# Comparativa de resultados
print("=" * 70)
print("COMPARATIVA FINAL: Modelos base vs Modelos con GridSearchCV")
print("=" * 70)

# DataFrame comparativo
comparativa = pd.DataFrame({
    'Modelo': ['Bagging', 'Random Forest', 'GradientBoosting', 'XGBoost'],
    'CV_Base': [cv_scores.mean(), rf_cv_scores.mean(), gb_cv_scores.mean(), xgb_cv_scores.mean()],
    'GridSearch_CV': [resultados_grid['Bagging']['best_score'],
                      resultados_grid['Random Forest']['best_score'],
                      resultados_grid['GradientBoosting']['best_score'],
                      resultados_grid['XGBoost']['best_score']],
    'Test': [resultados_grid['Bagging']['test_score'],
             resultados_grid['Random Forest']['test_score'],
             resultados_grid['GradientBoosting']['test_score'],
             resultados_grid['XGBoost']['test_score']]
})

print(comparativa)

COMPARATIVA FINAL: Modelos base vs Modelos con GridSearchCV
             Modelo   CV_Base  GridSearch_CV      Test
0           Bagging  0.752614       0.772025  0.805195
1     Random Forest  0.768199       0.781794  0.759740
2  GradientBoosting  0.760424       0.776903  0.766234
3           XGBoost  0.731733       0.781781  0.753247


1. Los modelos ensemble (Bagging, Random Forest, GradientBoosting, XGBoost) 
   ofrecen buenos resultados para este problema de clasificación binaria.

2. La hiperparametrización con GridSearchCV permite mejorar los resultados 
   encontrando los mejores hiperparámetros para cada modelo.

3. XGBoost y GradientBoosting suelen ser los modelos con mejor rendimiento 
   en problemas de clasificación tabular.

4. El dataset Pima Indians Diabetes es un problema clásico donde los modelos 
   ensemble obtienen accuracies alrededor del 75-80%.